# Partial Derivatives
**Topic:** Calculus for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** a partial derivative as the derivative with respect to one variable while all others are held fixed
- **Describe** how the gradient collects all partial derivatives into a single direction vector
- **Interpret** what it means geometrically to take a partial derivative of a 3D surface

> **Tip:** Start with the **3D surface** in the widget and click a point on it. Watch the two tangent lines appear: one in the x direction and one in the y direction. Those two slopes are the partial derivatives at that point.

---
## How we got here

In *04: The Chain Rule* we learned to differentiate composed functions. In *03: Derivatives* we differentiated functions of one variable. Every real ML model has more than one parameter. A linear regression with 10 features has 10 weights, each of which needs its own derivative. Partial derivatives extend the derivative idea to functions with many inputs, taking the derivative with respect to each one in turn.

---
## Why this matters for data science

Every weight in a machine learning model has its own partial derivative of the loss function. A model with 1,000 features has 1,000 partial derivatives. Collected together, those partial derivatives form the **gradient**: a vector that points in the direction of steepest increase in the loss. Gradient descent moves in the opposite direction, updating every weight simultaneously. Without partial derivatives there is no gradient, and without the gradient there is no training.

---
## Try it yourself

In [2]:
x_ax = np.linspace(-3.1, 3.1, 50)
y_ax = np.linspace(-3.1, 3.1, 50)
X_grid, Y_grid = np.meshgrid(x_ax, y_ax)

def f(x, y):    return np.sin(x) * np.cos(y)
def dfdx(x, y): return np.cos(x) * np.cos(y)
def dfdy(x, y): return -np.sin(x) * np.sin(y)

Z_grid = f(X_grid, Y_grid)

out = Output()

x_slider = FloatSlider(value=1.0, min=-3.1, max=3.1, step=0.1,
    description="x position:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"))

y_slider = FloatSlider(value=0.5, min=-3.1, max=3.1, step=0.1,
    description="y position:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"))

show_dd = Dropdown(
    options=["Both tangent lines", "∂f/∂x only  (y held fixed)", "∂f/∂y only  (x held fixed)"],
    value="Both tangent lines",
    description="Show:",
    style={"description_width": "55px"},
    layout=widgets.Layout(width="420px"))

def render(x0, y0, show):
    z0 = f(x0, y0)
    px = dfdx(x0, y0)
    py = dfdy(x0, y0)

    # Tangent in x direction: y held fixed at y0
    tx      = np.linspace(x0 - 1.2, x0 + 1.2, 40)
    tz_x    = z0 + px * (tx - x0)
    ty_x    = np.full_like(tx, y0)

    # Tangent in y direction: x held fixed at x0
    ty      = np.linspace(y0 - 1.2, y0 + 1.2, 40)
    tz_y    = z0 + py * (ty - y0)
    tx_y    = np.full_like(ty, x0)

    traces = [
        go.Surface(x=x_ax, y=y_ax, z=Z_grid,
            colorscale="Blues", opacity=0.65, showscale=False,
            name="f(x,y) = sin(x)·cos(y)"),
        go.Scatter3d(x=[x0], y=[y0], z=[z0], mode="markers",
            marker=dict(color="white", size=6,
                        line=dict(color=PALETTE["primary"], width=3)),
            name=f"({x0:.2f}, {y0:.2f})"),
    ]

    if "∂f/∂x" in show or show == "Both tangent lines":
        traces.append(go.Scatter3d(
            x=tx, y=ty_x, z=tz_x, mode="lines",
            line=dict(color=PALETTE["primary"], width=7),
            name=f"∂f/∂x = {px:.3f}  (y = {y0:.2f} fixed)"))

    if "∂f/∂y" in show or show == "Both tangent lines":
        traces.append(go.Scatter3d(
            x=tx_y, y=ty, z=tz_y, mode="lines",
            line=dict(color=PALETTE["secondary"], width=7),
            name=f"∂f/∂y = {py:.3f}  (x = {x0:.2f} fixed)"))

    layout = go.Layout(
        title=dict(
            text=(f"f({x0:.2f}, {y0:.2f}) = {z0:.3f}"
                  f"  |  ∂f/∂x = {px:.3f}"
                  f"  |  ∂f/∂y = {py:.3f}"),
            font=dict(size=FONT["size_title"]), x=0.5),
        paper_bgcolor=PALETTE["background"],
        scene=dict(
            xaxis=dict(title="x", backgroundcolor=PALETTE["background"], gridcolor="#DEE2E6"),
            yaxis=dict(title="y", backgroundcolor=PALETTE["background"], gridcolor="#DEE2E6"),
            zaxis=dict(title="f(x, y)", backgroundcolor=PALETTE["background"], gridcolor="#DEE2E6", range=[-1.5, 1.5]),
            bgcolor=PALETTE["background"],
        ),
        height=520,
        legend=dict(x=0, y=1, bgcolor="rgba(0,0,0,0)"),
    )
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces, layout=layout)))

def on_change(change): render(x_slider.value, y_slider.value, show_dd.value)
x_slider.observe(on_change, names="value")
y_slider.observe(on_change, names="value")
show_dd.observe(on_change, names="value")
display(VBox([x_slider, y_slider, show_dd, out]))
render(x_slider.value, y_slider.value, show_dd.value)

---
## What's happening?

When a function depends on multiple variables, say, loss(w₁, w₂), we cannot take "the" derivative because the function can change at different rates in different directions. A partial derivative takes the derivative in just one direction at a time, treating every other variable as a fixed constant.

We write the partial derivative of f with respect to x using the symbol ∂ (a curly d, different from the regular d to signal "partial"):

$$\frac{\partial f}{\partial x}$$

Read this as "the partial derivative of f with respect to x." To compute it, treat every other variable as a number and differentiate normally.

| Variable | Partial derivative | What it measures | ML interpretation |
|----------|-------------------|-----------------|------------------|
| Weight w₁ | ∂Loss/∂w₁ | How much the loss changes when w₁ changes slightly | The gradient component for w₁ |
| Weight w₂ | ∂Loss/∂w₂ | How much the loss changes when w₂ changes slightly | The gradient component for w₂ |
| Bias b | ∂Loss/∂b | How much the loss changes when the bias shifts | The gradient component for the bias |

### From partial derivatives to the gradient

The **gradient** is the vector that collects all partial derivatives together:

$$\nabla f = \left(\frac{\partial f}{\partial w_1},\; \frac{\partial f}{\partial w_2},\; \ldots,\; \frac{\partial f}{\partial w_n}\right)$$

The symbol ∇ is called "nabla" or "del." The gradient tells you the direction in which the function increases most steeply. To minimize the function, move in the opposite direction.

Return to the widget and pick a point on the surface. Confirm that the two slopes shown match the direction of steepest ascent when you look at the surface from above.

---
## Real-world example: Partial derivatives in linear regression with two features

A linear regression model with two features (x₁ = study hours, x₂ = sleep hours) has three parameters: w₁, w₂, and bias b. The loss surface lives in 4D, but we can visualize its bowl shape projected onto the w₁–w₂ plane. The chart shows the loss contours and the gradient vector at one point.

Notice:

- **Notice:** The gradient vector at any point on the contour map points perpendicular to the contour lines, directly toward the steepest uphill direction; gradient descent moves in the opposite direction
- **Notice:** The partial derivative ∂Loss/∂w₁ is the rate of change along the horizontal axis; ∂Loss/∂w₂ is along the vertical axis; together they form the gradient arrow
- **Notice:** At the center (the minimum), both partial derivatives equal zero and the gradient is the zero vector, no direction to move, which is exactly when training stops

> **Discussion question:** If ∂Loss/∂w₁ = −3.5 and ∂Loss/∂w₂ = +1.2 at the current weights, which weight would you increase and which would you decrease to reduce the loss? By how much would you change each if the learning rate is 0.01?

In [3]:
np.random.seed(42)
x1_rw = np.random.uniform(1, 8, 40)
x2_rw = np.random.uniform(5, 9, 40)
y_rw  = 2.1 * x1_rw + 1.4 * x2_rw + np.random.normal(0, 1.2, 40)

w1_ax = np.linspace(-1, 5, 55)
w2_ax = np.linspace(-1, 4, 55)

def mse_rw(w1, w2):
    return np.mean((y_rw - (w1 * x1_rw + w2 * x2_rw)) ** 2)

def grad_rw(w1, w2, eps=1e-4):
    dw1 = (mse_rw(w1 + eps, w2) - mse_rw(w1 - eps, w2)) / (2 * eps)
    dw2 = (mse_rw(w1, w2 + eps) - mse_rw(w1, w2 - eps)) / (2 * eps)
    return dw1, dw2

Z_rw  = np.array([[mse_rw(w1, w2) for w1 in w1_ax] for w2 in w2_ax])
_zmax = Z_rw.max() * 1.05   # fixed y-ceiling for slice charts

# Precompute background vector field on a coarse 7×7 grid (static)
_w1g = np.linspace(-0.5, 4.5, 7)
_w2g = np.linspace(-0.5, 3.5, 7)
_W1G, _W2G = np.meshgrid(_w1g, _w2g)
_W1f, _W2f = _W1G.ravel(), _W2G.ravel()
_ZGf  = np.array([mse_rw(w1, w2) for w1, w2 in zip(_W1f, _W2f)])
_GW1f = np.array([grad_rw(w1, w2)[0] for w1, w2 in zip(_W1f, _W2f)])
_GW2f = np.array([grad_rw(w1, w2)[1] for w1, w2 in zip(_W1f, _W2f)])
_mags = np.sqrt(_GW1f ** 2 + _GW2f ** 2)
_safe = _mags > 1e-8
_fu = np.where(_safe, _GW1f / _mags, 0.0)
_fv = np.where(_safe, _GW2f / _mags, 0.0)
_fw = np.zeros_like(_fu)

out_rw     = Output()
out_s1     = Output()
out_s2     = Output()
interp_lbl = widgets.HTML(value="", layout=widgets.Layout(padding="6px 0 0 0"))

w1_sl = FloatSlider(value=0.5, min=-1.0, max=5.0, step=0.1,
    description="Weight w₁:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"))

w2_sl = FloatSlider(value=0.5, min=-1.0, max=4.0, step=0.1,
    description="Weight w₂:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"))

def _interp(w1, w2, z0, dw1, dw2, mag):
    w1_dir = "increase" if dw1 < 0 else "decrease"
    w2_dir = "increase" if dw2 < 0 else "decrease"
    if abs(dw1) > abs(dw2) * 1.5:
        pull = "The study-hours weight (w₁) is responsible for most of the remaining error right now."
    elif abs(dw2) > abs(dw1) * 1.5:
        pull = "The sleep-hours weight (w₂) is responsible for most of the remaining error right now."
    else:
        pull = "Both weights contribute to the error about equally."
    if mag < 4:
        detail = ("The gradient is nearly zero — you are at or very near the minimum. "
                  "There is no direction that would improve the predictions, so training would stop here. "
                  "These weights are as good as this model can get.")
    elif z0 < 15:
        detail = (f"The orange arrow points uphill — away from the minimum. "
                  f"Gradient descent moves the other way: {w1_dir} w₁ and {w2_dir} w₂ slightly. "
                  f"You are close to the best-fit weights. {pull}")
    elif z0 < 60:
        detail = (f"The orange arrow shows the direction that raises error the fastest. "
                  f"Gradient descent steps the other way: {w1_dir} w₁ and {w2_dir} w₂. {pull}")
    else:
        detail = (f"You are on a steep part of the bowl, far from the minimum. "
                  f"The gradient magnitude is {mag:.0f} — a large step is needed. "
                  f"Gradient descent would {w1_dir} w₁ and {w2_dir} w₂ sharply. {pull}")
    return (
        f'<div style="background:#f8f9fa;border-left:4px solid {PALETTE["secondary"]};'
        f'padding:10px 14px;border-radius:4px;font-family:sans-serif;'
        f'font-size:14px;line-height:1.7;color:#333;">'
        f"<b>In plain English:</b><br>"
        f"Weights are w₁ = {w1:.2f}, w₂ = {w2:.2f}. "
        f"The model's average squared error is <b>{z0:.1f}</b>. "
        f"{detail}</div>"
    )

def render_rw(w1, w2):
    z0       = mse_rw(w1, w2)
    dw1, dw2 = grad_rw(w1, w2)
    mag      = np.sqrt(dw1 ** 2 + dw2 ** 2)
    nd1      = dw1 / mag if mag > 1e-8 else 0.0
    nd2      = dw2 / mag if mag > 1e-8 else 0.0

    # ── 3D loss bowl ─────────────────────────────────────────────────────
    with out_rw:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(
            data=[
                go.Surface(x=w1_ax, y=w2_ax, z=Z_rw,
                    colorscale="Blues", opacity=0.6, showscale=False,
                    name="MSE surface"),
                go.Cone(x=_W1f, y=_W2f, z=_ZGf, u=_fu, v=_fv, w=_fw,
                    colorscale=[[0, PALETTE["muted"]], [1, PALETTE["muted"]]],
                    showscale=False, sizemode="absolute", sizeref=0.28,
                    anchor="tail", opacity=0.35, name="Gradient field"),
                go.Scatter3d(x=[w1], y=[w2], z=[z0], mode="markers",
                    marker=dict(color="white", size=7,
                                line=dict(color=PALETTE["primary"], width=3)),
                    name=f"MSE = {z0:.1f}"),
                go.Cone(x=[w1], y=[w2], z=[z0], u=[nd1], v=[nd2], w=[0.0],
                    colorscale=[[0, PALETTE["secondary"]], [1, PALETTE["secondary"]]],
                    showscale=False, sizemode="absolute", sizeref=0.55,
                    anchor="tail", name=f"∇L = ({dw1:.2f}, {dw2:.2f})"),
            ],
            layout=go.Layout(
                title=dict(
                    text=f"MSE = {z0:.1f}  |  ∇L = ({dw1:.2f}, {dw2:.2f})  |  ‖∇L‖ = {mag:.2f}",
                    font=dict(size=14), x=0.5),
                paper_bgcolor=PALETTE["background"],
                scene=dict(
                    xaxis=dict(title="w₁ (study hours)",
                               backgroundcolor=PALETTE["background"], gridcolor="#DEE2E6"),
                    yaxis=dict(title="w₂ (sleep hours)",
                               backgroundcolor=PALETTE["background"], gridcolor="#DEE2E6"),
                    zaxis=dict(title="MSE",
                               backgroundcolor=PALETTE["background"], gridcolor="#DEE2E6"),
                    bgcolor=PALETTE["background"]),
                height=500, legend=dict(x=0, y=1, bgcolor="rgba(0,0,0,0)"),
            )
        )))

    # ── Slice 1: vary w₁, hold w₂ fixed ─────────────────────────────────
    curve1 = np.array([mse_rw(w, w2) for w in w1_ax])
    t1     = np.linspace(w1 - 1.2, w1 + 1.2, 40)
    tang1  = z0 + dw1 * (t1 - w1)
    l1 = base_layout(
        title=f"Slice: w₂ = {w2:.2f} fixed  →  ∂L/∂w₁ = {dw1:.2f}",
        xaxis_title="w₁", yaxis_title="Loss")
    l1.update(height=300, yaxis=dict(range=[0, _zmax]))
    with out_s1:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(
            data=[
                go.Scatter(x=w1_ax, y=curve1, mode="lines",
                    line=dict(color=PALETTE["muted"], width=2.5),
                    name="L(w₁, w₂ fixed)"),
                go.Scatter(x=t1, y=tang1, mode="lines",
                    line=dict(color=PALETTE["primary"], width=2, dash="dash"),
                    name=f"tangent  slope = {dw1:.2f}"),
                go.Scatter(x=[w1], y=[z0], mode="markers",
                    marker=dict(color=PALETTE["primary"], size=10),
                    name="current w₁"),
            ],
            layout=l1
        )))

    # ── Slice 2: vary w₂, hold w₁ fixed ─────────────────────────────────
    curve2 = np.array([mse_rw(w1, w) for w in w2_ax])
    t2     = np.linspace(w2 - 1.2, w2 + 1.2, 40)
    tang2  = z0 + dw2 * (t2 - w2)
    l2 = base_layout(
        title=f"Slice: w₁ = {w1:.2f} fixed  →  ∂L/∂w₂ = {dw2:.2f}",
        xaxis_title="w₂", yaxis_title="Loss")
    l2.update(height=300, yaxis=dict(range=[0, _zmax]))
    with out_s2:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(
            data=[
                go.Scatter(x=w2_ax, y=curve2, mode="lines",
                    line=dict(color=PALETTE["muted"], width=2.5),
                    name="L(w₁ fixed, w₂)"),
                go.Scatter(x=t2, y=tang2, mode="lines",
                    line=dict(color=PALETTE["secondary"], width=2, dash="dash"),
                    name=f"tangent  slope = {dw2:.2f}"),
                go.Scatter(x=[w2], y=[z0], mode="markers",
                    marker=dict(color=PALETTE["secondary"], size=10),
                    name="current w₂"),
            ],
            layout=l2
        )))

    interp_lbl.value = _interp(w1, w2, z0, dw1, dw2, mag)

def on_change_rw(change): render_rw(w1_sl.value, w2_sl.value)
w1_sl.observe(on_change_rw, names="value")
w2_sl.observe(on_change_rw, names="value")
display(VBox([w1_sl, w2_sl, out_rw, HBox([out_s1, out_s2]), interp_lbl]))
render_rw(w1_sl.value, w2_sl.value)

### Partial derivatives in ML models

| Variable | Partial derivative | Plain English meaning | ML interpretation |
|----------|-------------------|-----------------------|------------------|
| w₁ | ∂Loss/∂w₁ | Loss changes this fast when w₁ nudges up | Update direction for feature 1's weight |
| Bias b | ∂Loss/∂b | Loss changes this fast when the intercept shifts | Update direction for the bias term |
| Layer weights W | ∂Loss/∂W | Loss gradient with respect to a weight matrix | The matrix of updates sent to that layer |
| Activation a | ∂Loss/∂a | Loss changes this fast when the pre-activation changes | Propagated error signal in backprop |

---
## Key takeaway

> **A partial derivative measures how fast a function changes in one direction while everything else stays frozen; the gradient stacks all partial derivatives into a vector pointing toward steepest increase, and gradient descent follows it downhill.**

---
*Next up: The Gradient & Gradient Descent — how the gradient becomes the compass that guides a model's parameters toward the minimum loss*